# Brachiosaurus Training Notebook

Train a brachiosaurus to walk on four legs and reach food using reinforcement learning.

**Training Stages:**
1. **Balance** - Learn to stand on four legs without falling
2. **Locomotion** - Coordinated quadrupedal walking
3. **Food Reach** - Walk to food and reach with neck

**Supported Algorithms:** PPO, SAC

This notebook loads all configs from the TOML files in `configs/brachiosaurus/` and supports
the full 3-stage curriculum with either algorithm.

## 1. Setup & Installation

In [ ]:
# Install dependencies (Colab auto-detected; no-op locally)
import importlib
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")

if IN_COLAB:
    # Configure headless rendering for MuJoCo (must happen before mujoco import)
    os.environ["MUJOCO_GL"] = "egl"
    NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
    if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
        os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
        with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
            f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

    # Install packages only if not already present
    if importlib.util.find_spec("mujoco") is None:
        get_ipython().system(
            'pip install -q mujoco>=3.0.0 gymnasium>=0.29.0 "stable-baselines3[extra]>=2.2.0" mediapy matplotlib'
        )
    import pathlib
    import subprocess

    repo_dir = pathlib.Path("/content/mesozoic-labs")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/kuds/mesozoic-labs.git", str(repo_dir)], check=True)
    if importlib.util.find_spec("environments") is None:
        get_ipython().system("pip install -q -e /content/mesozoic-labs")
    print("Colab setup complete (EGL rendering enabled).")
else:
    print("Running locally.")

In [ ]:
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Add repo root to path (works both locally from notebooks/ and in Colab)
if IN_COLAB:
    repo_root = Path("/content/mesozoic-labs")
else:
    repo_root = Path("..").resolve()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import gymnasium as gym
import mujoco

print(f"MuJoCo version: {mujoco.__version__}")
print(f"Gymnasium version: {gym.__version__}")
print(f"Repo root: {repo_root}")

## 2. Configuration

Choose your algorithm and training parameters here. All reward weights and
hyperparameters are loaded from the TOML configs in `configs/brachiosaurus/`.

In [ ]:
# ============================================================
# USER CONFIGURATION - Modify these values as needed
# ============================================================

ALGORITHM = "PPO"  # "PPO" or "SAC"
N_ENVS = 4  # Number of parallel environments
SEED = 42  # Random seed for reproducibility
QUICK_TEST = True  # Set to False for full training runs
USE_GOOGLE_DRIVE = False  # Set to True to save logs/models to Google Drive (Colab only)
VERBOSE = 1  # 0=eval results only, 1=training stats + progress bar (default), 2=debug

# ============================================================
# Load configs from TOML files
# ============================================================
from environments.shared.config import load_all_stages, save_stage_config

STAGE_CONFIGS = load_all_stages("brachiosaurus")

algo_key = "ppo_kwargs" if ALGORITHM == "PPO" else "sac_kwargs"

print(f"Algorithm: {ALGORITHM}")
print(f"Parallel envs: {N_ENVS}")
print(f"Quick test: {QUICK_TEST}")
print(f"Verbose: {VERBOSE}")
print(f"Google Drive storage: {USE_GOOGLE_DRIVE}")
print()
for stage, config in STAGE_CONFIGS.items():
    cur = config.get("curriculum_kwargs", {})
    ts = 50_000 if QUICK_TEST else cur.get("timesteps", 1_000_000)
    print(f"Stage {stage}: {config['name']} - {config['description']}")
    print(f"  Timesteps: {ts:,}")
    print(f"  {ALGORITHM} hyperparams: {config[algo_key]}")
    print()

In [ ]:
# ============================================================
# Storage Configuration
# ============================================================
# When USE_GOOGLE_DRIVE is True and running in Colab, logs and
# models are saved to Google Drive so they persist across sessions.
# Otherwise, everything is saved to the local filesystem.

if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    LOG_BASE = Path("/content/drive/MyDrive/mesozoic-labs/logs")
    LOG_BASE.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted. Logs will be saved to: {LOG_BASE}")
elif USE_GOOGLE_DRIVE and not IN_COLAB:
    print("Warning: USE_GOOGLE_DRIVE is True but not running in Colab. Using local storage.")
    LOG_BASE = repo_root / "logs"
    print(f"Logs will be saved to: {LOG_BASE}")
else:
    LOG_BASE = repo_root / "logs"
    print(f"Logs will be saved to: {LOG_BASE}")

# Create a single run directory for all stages, organised by species then datetime
SPECIES = "brachiosaurus"
RUN_DIR = LOG_BASE / SPECIES / f"{ALGORITHM.lower()}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Run directory: {RUN_DIR}")

## 3. Explore the Environment

In [ ]:
from environments.brachiosaurus.envs.brachio_env import BrachioEnv

env = BrachioEnv()

print("Environment loaded successfully!")
print(f"\nObservation space: {env.observation_space}")
print(f"  Shape: {env.observation_space.shape}")
print(f"\nAction space: {env.action_space}")
print(f"  Shape: {env.action_space.shape}")
print(f"  Range: [{env.action_space.low[0]}, {env.action_space.high[0]}]")

model_mj = env.model
print("\nModel Information:")
print(f"  Bodies: {model_mj.nbody}")
print(f"  Joints: {model_mj.njnt}")
print(f"  Actuators: {model_mj.nu}")
print(f"  Sensors: {model_mj.nsensor}")
print(f"  Total DOF: {model_mj.nv}")
print(f"  Total mass: {sum(model_mj.body_mass):.2f} kg")

print("\nActuators:")
for i in range(model_mj.nu):
    name = mujoco.mj_id2name(model_mj, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
    print(f"  [{i:2d}] {name}")

env.close()

In [ ]:
# Run random episodes to establish a baseline
env = BrachioEnv()
n_episodes = 5
episode_rewards = []
episode_lengths = []

for ep in range(n_episodes):
    obs, info = env.reset(seed=ep)
    total_reward = 0
    step = 0
    while True:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        step += 1
        if terminated or truncated:
            break
    episode_rewards.append(total_reward)
    episode_lengths.append(step)
    print(f"  Episode {ep + 1}: reward={total_reward:.2f}, length={step}")

print("\nRandom policy baseline:")
print(f"  Avg reward: {np.mean(episode_rewards):.2f} +/- {np.std(episode_rewards):.2f}")
print(f"  Avg length: {np.mean(episode_lengths):.1f} +/- {np.std(episode_lengths):.1f}")
env.close()

## 4. Training Infrastructure

In [ ]:
import time

from stable_baselines3 import PPO, SAC
from stable_baselines3.common.callbacks import BaseCallback, CallbackList, CheckpointCallback, EvalCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.utils import get_linear_fn, get_schedule_fn, set_random_seed
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

ALGO_CLASS = {"PPO": PPO, "SAC": SAC}

try:
    import mediapy

    _HAS_MEDIAPY = True
except ImportError:
    _HAS_MEDIAPY = False
    print("mediapy not installed. Videos will be skipped. Install with: pip install mediapy")


# ============================================================
# Diagnostics Callback - logs reward components, obs/action
# stats, VecNormalize state, and plateau warnings to TensorBoard
# ============================================================
class DiagnosticsCallback(BaseCallback):
    """Logs per-component reward breakdowns and training diagnostics to TensorBoard.

    Tracked metrics (under ``diagnostics/`` in TensorBoard):
      - Per-component rewards: reward_forward, reward_alive, reward_energy, etc.
      - Environment state: forward_vel, prey_distance, pelvis_height
      - Observation statistics: mean, std, max absolute value
      - Action statistics: mean, std
      - VecNormalize running variance for observations and returns
      - Reward plateau detection with console warnings
    """

    REWARD_KEYS = [
        "reward_forward",
        "reward_alive",
        "reward_energy",
        "reward_tail",
        "reward_strike",
        "reward_approach",
        "reward_gait",
        "reward_neck",
        "reward_food_reach",
        "reward_bite",
    ]
    INFO_KEYS = [
        "forward_vel",
        "prey_distance",
        "pelvis_height",
        "jaw_distance",
        "tail_instability",
    ]

    def __init__(self, plateau_window=10, plateau_threshold=1.0, verbose=0):
        super().__init__(verbose)
        self.plateau_window = plateau_window
        self.plateau_threshold = plateau_threshold
        self._step_infos = {k: [] for k in self.REWARD_KEYS + self.INFO_KEYS}
        self._rollout_ep_rewards = []

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            for key in self.REWARD_KEYS + self.INFO_KEYS:
                if key in info:
                    self._step_infos[key].append(float(info[key]))
        return True

    def _on_rollout_end(self) -> None:
        # Per-component reward breakdown
        for key, values in self._step_infos.items():
            if values:
                self.logger.record(f"diagnostics/{key}", np.mean(values))
        self._step_infos = {k: [] for k in self.REWARD_KEYS + self.INFO_KEYS}

        # Observation statistics from rollout buffer
        if hasattr(self.model, "rollout_buffer") and self.model.rollout_buffer.observations is not None:
            obs = self.model.rollout_buffer.observations
            self.logger.record("diagnostics/obs_mean", float(np.mean(obs)))
            self.logger.record("diagnostics/obs_std", float(np.std(obs)))
            self.logger.record("diagnostics/obs_max_abs", float(np.max(np.abs(obs))))

        # Action statistics from rollout buffer
        if hasattr(self.model, "rollout_buffer") and self.model.rollout_buffer.actions is not None:
            acts = self.model.rollout_buffer.actions
            self.logger.record("diagnostics/action_mean", float(np.mean(acts)))
            self.logger.record("diagnostics/action_std", float(np.std(acts)))

        # VecNormalize running statistics
        env = self.training_env
        if hasattr(env, "obs_rms"):
            self.logger.record("diagnostics/vecnorm_obs_var_mean", float(np.mean(env.obs_rms.var)))
        if hasattr(env, "ret_rms"):
            self.logger.record("diagnostics/vecnorm_ret_var", float(np.mean(env.ret_rms.var)))

        # Plateau detection from completed episodes
        ep_rewards = [info["episode"]["r"] for info in self.locals.get("infos", []) if "episode" in info]
        if ep_rewards:
            self._rollout_ep_rewards.append(np.mean(ep_rewards))
            if len(self._rollout_ep_rewards) >= self.plateau_window:
                recent = self._rollout_ep_rewards[-self.plateau_window :]
                variation = max(recent) - min(recent)
                self.logger.record("diagnostics/reward_variation", variation)
                if variation < self.plateau_threshold:
                    print(
                        f"\n*** PLATEAU WARNING: Reward variation over last "
                        f"{self.plateau_window} rollouts is only {variation:.4f}. "
                        f"Consider adjusting learning rate or stopping. ***\n"
                    )


def make_env(stage, rank, seed=0):
    """Create a single environment instance."""

    def _init():
        env_kwargs = STAGE_CONFIGS[stage]["env_kwargs"].copy()
        env = BrachioEnv(**env_kwargs)
        env = Monitor(env)
        env.reset(seed=seed + rank)
        return env

    set_random_seed(seed)
    return _init


def create_vec_env(stage, n_envs=N_ENVS, seed=SEED, vecnorm_path=None):
    """Create vectorized environment with observation/reward normalization.

    If vecnorm_path is provided, loads normalization stats from a prior stage
    so the policy sees consistently-scaled observations across curriculum stages.
    """
    env = DummyVecEnv([make_env(stage, i, seed) for i in range(n_envs)])
    if vecnorm_path and Path(vecnorm_path).exists():
        env = VecNormalize.load(vecnorm_path, env)
        env.training = True
        print(f"  Loaded VecNormalize stats from prior stage: {vecnorm_path}")
    else:
        env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.0, clip_reward=10.0)
    return env


def get_algo_kwargs(stage):
    """Get algorithm-specific hyperparameters for a stage."""
    config = STAGE_CONFIGS[stage]
    if ALGORITHM == "PPO":
        return config["ppo_kwargs"].copy()
    else:
        return config["sac_kwargs"].copy()


def record_stage_video(model, stage, stage_dir, max_steps=1000):
    """Record and save a video of the trained policy for a given stage."""
    if not _HAS_MEDIAPY:
        print(f"Skipping video for stage {stage} (mediapy not installed).")
        return

    env_kwargs = STAGE_CONFIGS[stage]["env_kwargs"].copy()
    render_env = BrachioEnv(render_mode="rgb_array", **env_kwargs)

    obs, _ = render_env.reset(seed=SEED + 2000 + stage)
    frames = []
    episode_reward = 0.0

    for _ in range(max_steps):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = render_env.step(action)
        frames.append(render_env.render())
        episode_reward += reward
        if terminated or truncated:
            break

    render_env.close()

    video_path = str(Path(stage_dir) / f"brachiosaurus_{ALGORITHM.lower()}_stage{stage}.mp4")
    mediapy.write_video(video_path, frames, fps=50)
    print(f"Stage {stage} video: reward={episode_reward:.2f} | {len(frames)} frames")
    print(f"  Saved to: {video_path}")
    mediapy.show_video(frames, fps=50)


def train_stage(
    stage,
    timesteps,
    load_path=None,
    run_dir=None,
    vecnorm_path=None,
):
    """Train a single curriculum stage.

    All stages are saved under the shared run_dir as stage subdirectories.

    Returns (model, best_model_path, stage_dir, vecnorm_save_path, stage_results).
    """
    stage_start = time.time()

    config = STAGE_CONFIGS[stage]
    AlgoClass = ALGO_CLASS[ALGORITHM]
    algo_kwargs = get_algo_kwargs(stage)
    algo_kwargs["verbose"] = VERBOSE

    # Extract policy_kwargs from TOML config, falling back to default net_arch
    policy_kwargs = algo_kwargs.pop("policy_kwargs", {"net_arch": [256, 256]})

    # Handle learning_rate_end: create a linear schedule that decays from
    # learning_rate down to learning_rate_end over the course of training
    lr_end = algo_kwargs.pop("learning_rate_end", None)
    if lr_end is not None:
        lr_start = algo_kwargs["learning_rate"]
        algo_kwargs["learning_rate"] = get_linear_fn(lr_start, lr_end, 1.0)

    # Directories - each stage gets a subdirectory within the run directory
    if run_dir is None:
        run_dir = repo_root / "logs"
    stage_dir = Path(run_dir) / f"stage{stage}"
    stage_dir.mkdir(parents=True, exist_ok=True)
    model_dir = stage_dir / "models"
    model_dir.mkdir(exist_ok=True)
    algo_kwargs["tensorboard_log"] = str(stage_dir / "tensorboard")

    print(f"{'=' * 60}")
    print(f"Stage {stage}: {config['name']} ({ALGORITHM})")
    print(f"Description: {config['description']}")
    print(f"Timesteps: {timesteps:,}")
    print(f"Log dir: {stage_dir}")
    if vecnorm_path:
        print(f"VecNormalize from prior stage: {vecnorm_path}")
    print(f"{'=' * 60}")

    # Save reward weights and hyperparameters for reproducibility
    cfg_path = save_stage_config(
        stage_dir,
        stage,
        config,
        ALGORITHM,
        extra={"seed": SEED, "n_envs": N_ENVS, "timesteps": timesteps},
    )
    print(f"Stage config saved to: {cfg_path}")

    # Environments (carry over normalization stats from prior stage)
    train_env = create_vec_env(stage, vecnorm_path=vecnorm_path)
    eval_env = create_vec_env(stage, n_envs=1, seed=SEED + 1000, vecnorm_path=vecnorm_path)

    # Create or load model
    if load_path:
        print(f"Loading model from: {load_path}")
        model = AlgoClass.load(load_path, env=train_env)
        # SB3 expects schedule functions, not raw floats
        if callable(algo_kwargs["learning_rate"]):
            model.learning_rate = algo_kwargs["learning_rate"]
        else:
            model.learning_rate = get_schedule_fn(algo_kwargs["learning_rate"])
        if ALGORITHM == "PPO":
            model.ent_coef = algo_kwargs["ent_coef"]
            model.clip_range = get_schedule_fn(algo_kwargs["clip_range"])
    else:
        model = AlgoClass(
            "MlpPolicy",
            train_env,
            policy_kwargs=policy_kwargs,
            **algo_kwargs,
        )

    # Callbacks
    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=str(model_dir),
        log_path=str(stage_dir),
        eval_freq=max(5000 // N_ENVS, 1),
        n_eval_episodes=20,
        deterministic=True,
        verbose=max(VERBOSE, 1),
    )
    checkpoint_callback = CheckpointCallback(
        save_freq=max(25000 // N_ENVS, 1),
        save_path=str(model_dir),
        name_prefix=f"stage{stage}",
        save_vecnormalize=True,
    )
    diagnostics_callback = DiagnosticsCallback(
        plateau_window=10,
        plateau_threshold=1.0,
    )

    # Train
    model.learn(
        total_timesteps=timesteps,
        callback=CallbackList([eval_callback, checkpoint_callback, diagnostics_callback]),
        progress_bar=VERBOSE >= 1,
    )

    # Save final model and VecNormalize stats
    final_path = model_dir / f"stage{stage}_final"
    model.save(str(final_path))
    vecnorm_save_path = str(final_path) + "_vecnorm.pkl"
    train_env.save(vecnorm_save_path)
    print(f"\nFinal model saved to: {final_path}.zip")
    print(f"VecNormalize stats saved to: {vecnorm_save_path}")

    # Use best model for video recording and next-stage loading
    best_model_zip = model_dir / "best_model.zip"
    if best_model_zip.exists():
        best_path = model_dir / "best_model"
        model = AlgoClass.load(str(best_path), env=train_env)
        output_path = str(best_path)
        print(f"Loaded best model for video/next-stage: {best_path}.zip")
    else:
        output_path = str(final_path)

    # Quick evaluation
    mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=20)
    print(f"Eval: mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")

    train_env.close()
    eval_env.close()

    stage_duration = time.time() - stage_start
    stage_results = {
        "stage": stage,
        "name": config["name"],
        "description": config["description"],
        "timesteps": timesteps,
        "duration_seconds": stage_duration,
        "mean_reward": float(mean_reward),
        "std_reward": float(std_reward),
        "model_path": output_path,
        "vecnorm_path": vecnorm_save_path,
    }

    return model, output_path, stage_dir, vecnorm_save_path, stage_results


def _format_duration(seconds):
    """Format seconds into a human-readable string."""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    if h > 0:
        return f"{h}h {m}m {s}s"
    elif m > 0:
        return f"{m}m {s}s"
    return f"{s}s"


def _format_duration_hms(seconds):
    """Format seconds as H:MM:SS."""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h}:{m:02d}:{s:02d}"


def write_training_summary(run_dir, stage_results_list, species="Brachiosaurus"):
    """Write a training summary text file to the run directory."""
    summary_path = Path(run_dir) / "training_summary.txt"
    total_duration = sum(r["duration_seconds"] for r in stage_results_list)

    lines = [
        "Mesozoic Labs Training Summary",
        "=" * 50,
        "",
        f"Species:        {species}",
        f"Algorithm:      {ALGORITHM}",
        f"Date:           {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"Seed:           {SEED}",
        f"Quick test:     {QUICK_TEST}",
        f"Parallel envs:  {N_ENVS}",
        f"Run directory:  {run_dir}",
        "",
    ]

    for r in stage_results_list:
        lines.extend(
            [
                f"Stage {r['stage']}: {r['name']}",
                f"  Description:    {r['description']}",
                f"  Timesteps:      {r['timesteps']:,}",
                f"  Duration:       {_format_duration(r['duration_seconds'])}",
                f"  Eval reward:    {r['mean_reward']:.2f} +/- {r['std_reward']:.2f}",
                f"  Best model:     {r['model_path']}.zip",
                "",
            ]
        )

    lines.extend(
        [
            "-" * 50,
            f"Total training time: {_format_duration(total_duration)}",
        ]
    )

    summary_text = "\n".join(lines) + "\n"
    summary_path.write_text(summary_text)
    print(f"\nTraining summary saved to: {summary_path}")
    print(summary_text)


def save_results_json(stage_results_list, species="brachiosaurus"):
    """Save a summary.json to results/<species>/<algorithm>/ in the repo.

    This creates a machine-readable record of the training run that can be
    used to auto-generate the README results table and website content.
    """
    import json

    algo_lower = ALGORITHM.lower()
    results_dir = repo_root / "results" / species / algo_lower
    results_dir.mkdir(parents=True, exist_ok=True)

    total_duration = sum(r["duration_seconds"] for r in stage_results_list)
    total_timesteps = sum(r["timesteps"] for r in stage_results_list)
    final_result = stage_results_list[-1]

    stages = {}
    for r in stage_results_list:
        stages[str(r["stage"])] = {
            "name": r["name"],
            "timesteps": r["timesteps"],
            "avg_reward": round(r["mean_reward"], 2),
            "std_reward": round(r["std_reward"], 2),
            "training_time_seconds": round(r["duration_seconds"], 1),
            "training_time": _format_duration_hms(r["duration_seconds"]),
        }

    summary = {
        "species": species,
        "algorithm": ALGORITHM,
        "hardware": "Google Colab T4 GPU",
        "seed": SEED,
        "date": datetime.now().strftime("%Y-%m-%d"),
        "stages": stages,
        "total_timesteps": total_timesteps,
        "total_training_time_seconds": round(total_duration, 1),
        "total_training_time": _format_duration_hms(total_duration),
        "final_avg_reward": round(final_result["mean_reward"], 2),
    }

    summary_path = results_dir / "summary.json"
    summary_path.write_text(json.dumps(summary, indent=2) + "\n")
    print(f"\nResults summary saved to: {summary_path}")
    return summary_path


print(f"Training infrastructure ready. Algorithm: {ALGORITHM}")
print("DiagnosticsCallback enabled: per-component rewards, obs/action stats,")
print("VecNormalize tracking, and plateau detection will log to TensorBoard.")

## 5. Stage 1: Balance

The brachiosaurus learns to stand on four legs without toppling. No forward
velocity reward — just a strong alive bonus and gait stability.

In [ ]:
cur1 = STAGE_CONFIGS[1].get("curriculum_kwargs", {})
timesteps_1 = 50_000 if QUICK_TEST else cur1.get("timesteps", 1_000_000)

model_1, path_1, dir_1, vecnorm_1, results_1 = train_stage(stage=1, timesteps=timesteps_1, run_dir=RUN_DIR)

In [ ]:
record_stage_video(model_1, stage=1, stage_dir=dir_1)

## 6. Stage 2: Locomotion

Starting from the Stage 1 checkpoint, the brachiosaurus learns coordinated
quadrupedal walking.

In [ ]:
cur2 = STAGE_CONFIGS[2].get("curriculum_kwargs", {})
timesteps_2 = 50_000 if QUICK_TEST else cur2.get("timesteps", 2_000_000)

model_2, path_2, dir_2, vecnorm_2, results_2 = train_stage(
    stage=2, timesteps=timesteps_2, load_path=path_1, run_dir=RUN_DIR, vecnorm_path=vecnorm_1
)

In [ ]:
record_stage_video(model_2, stage=2, stage_dir=dir_2)

## 7. Stage 3: Food Reach

The brachiosaurus learns to walk toward food and reach it with its long neck.

In [ ]:
cur3 = STAGE_CONFIGS[3].get("curriculum_kwargs", {})
timesteps_3 = 50_000 if QUICK_TEST else cur3.get("timesteps", 3_000_000)

model_3, path_3, dir_3, vecnorm_3, results_3 = train_stage(
    stage=3, timesteps=timesteps_3, load_path=path_2, run_dir=RUN_DIR, vecnorm_path=vecnorm_2
)

In [ ]:
record_stage_video(model_3, stage=3, stage_dir=dir_3)

## 8. Evaluate Final Policy

In [ ]:
def evaluate_trained_policy(model, stage, n_episodes=10):
    """Evaluate a trained policy with per-component reward breakdown."""
    env_kwargs = STAGE_CONFIGS[stage]["env_kwargs"].copy()
    env = BrachioEnv(**env_kwargs)

    episode_rewards = []
    episode_lengths = []
    component_totals = {}

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=ep + 100)
        total_reward = 0
        step = 0
        ep_components = {}
        while True:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            step += 1
            for key in info:
                if key.startswith("reward_"):
                    ep_components[key] = ep_components.get(key, 0.0) + info[key]
            if terminated or truncated:
                break
        episode_rewards.append(total_reward)
        episode_lengths.append(step)
        for key, val in ep_components.items():
            component_totals.setdefault(key, []).append(val)
        print(f"  Episode {ep + 1}: reward={total_reward:.2f}, length={step}")

    env.close()
    print("\nResults:")
    print(f"  Mean reward: {np.mean(episode_rewards):.2f} +/- {np.std(episode_rewards):.2f}")
    print(f"  Mean length: {np.mean(episode_lengths):.1f} +/- {np.std(episode_lengths):.1f}")
    if component_totals:
        print("\nReward component breakdown (mean per episode):")
        for key, vals in sorted(component_totals.items()):
            print(f"  {key}: {np.mean(vals):.2f} +/- {np.std(vals):.2f}")
    return episode_rewards, episode_lengths


print(f"Evaluating final Stage 3 policy ({ALGORITHM})...")
rewards_3, lengths_3 = evaluate_trained_policy(model_3, stage=3)

## 9. Training Curves

In [ ]:
def plot_training_curve(stage_dir, stage, algo_name):
    """Plot the evaluation reward curve from a training run."""
    eval_log = Path(stage_dir) / "evaluations.npz"
    if not eval_log.exists():
        print(f"No evaluation log found for stage {stage}.")
        return

    data = np.load(eval_log)
    timesteps = data["timesteps"]
    results = data["results"]
    mean_rewards = np.mean(results, axis=1)
    std_rewards = np.std(results, axis=1)

    plt.plot(timesteps, mean_rewards, label=f"Stage {stage}: {STAGE_CONFIGS[stage]['name']}")
    plt.fill_between(timesteps, mean_rewards - std_rewards, mean_rewards + std_rewards, alpha=0.2)


plt.figure(figsize=(12, 5))
for stage_num, stage_dir in [(1, dir_1), (2, dir_2), (3, dir_3)]:
    plot_training_curve(stage_dir, stage_num, ALGORITHM)

plt.xlabel("Timesteps")
plt.ylabel("Mean Reward")
plt.title(f"Brachiosaurus {ALGORITHM} - Curriculum Training Progress")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Record All Stage Videos

Re-record videos for all three training stages. This is useful if you want to
regenerate videos without re-running training.

In [ ]:
for stage, model, stage_dir in [(1, model_1, dir_1), (2, model_2, dir_2), (3, model_3, dir_3)]:
    print(f"\n{'=' * 40}")
    print(f"Recording Stage {stage}: {STAGE_CONFIGS[stage]['name']}")
    print(f"{'=' * 40}")
    record_stage_video(model, stage=stage, stage_dir=stage_dir)

## 11. Cleanup

In [ ]:
write_training_summary(RUN_DIR, [results_1, results_2, results_3])
save_results_json([results_1, results_2, results_3], species="brachiosaurus")

print("Training complete!")
print(f"\nAlgorithm: {ALGORITHM}")
print(f"Run directory: {RUN_DIR}")
print(f"Stage 1 model: {path_1}.zip")
print(f"Stage 2 model: {path_2}.zip")
print(f"Stage 3 model: {path_3}.zip")
print("\nTo run the other algorithm, change ALGORITHM at the top and re-run all cells.")